In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
import random

In [0]:
spark = SparkSession.builder.appName("DataAnalysis_PySpark").getOrCreate()

In [0]:
# Create schema

schema = StructType([
    StructField("altitude", ArrayType(DoubleType()), True),
    StructField("gender", StringType(), True),
    StructField("heart_rate", ArrayType(LongType()), True),
    StructField("id", LongType(), True),
    StructField("latitude", ArrayType(DoubleType()), True),
    StructField("longitude", ArrayType(DoubleType()), True),
    StructField("speed", ArrayType(DoubleType()), True),
    StructField("sport", StringType(), True),
    StructField("timestamp", ArrayType(LongType()), True),
    StructField("url", StringType(), True),
    StructField("userId", LongType(), True)
])

# Generate Data

def generate_data():
    sports = ["running", "cycling", "swimming", "hiking"]
    genders = ["male", "female", "other"]

    for x in range(201):
        yield(
           [random.uniform(0, 3000) for _ in range(5)],  # altitude
            random.choice(genders),                      # gender
            [random.randint(60, 180) for _ in range(5)], # heart_rate
            x,                                           # id
            [random.uniform(-90, 90) for _ in range(5)], # latitude
            [random.uniform(-180, 180) for _ in range(5)],# longitude
            [random.uniform(0, 50) for _ in range(5)],   # speed
            random.choice(sports),                      # sport
            [random.randint(1_600_000_000, 1_700_000_000) for _ in range(5)], # timestamp
            f"https://example.com/user_{x}",            # url
            random.randint(1, 1000)                     # userId 
        )

data = list(generate_data())
df = spark.createDataFrame(data, schema=schema)
df.show(5,truncate=True)



+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+-------+--------------------+--------------------+------+
|            altitude|gender|          heart_rate| id|            latitude|           longitude|               speed|  sport|           timestamp|                 url|userId|
+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+-------+--------------------+--------------------+------+
|[1045.00502653444...| other|[162, 108, 66, 14...|  0|[49.1390526449582...|[166.576502984078...|[0.59193083225480...|running|[1630938389, 1637...|https://example.c...|   332|
|[2946.96314412445...| other|[121, 98, 168, 13...|  1|[-33.076971821503...|[134.795814986874...|[23.1945693137552...|running|[1631979990, 1649...|https://example.c...|   567|
|[349.632634314160...|  male|[177, 139, 150, 1...|  2|[85.8509495785931...|[-38.668059705926...|[38.3069744362893...|running|

In [0]:
# Create a temp view of df
df.createOrReplaceTempView('df_tbl')
spark.sql("SELECT * FROM df_tbl ").show(5)


+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+-------+--------------------+--------------------+------+
|            altitude|gender|          heart_rate| id|            latitude|           longitude|               speed|  sport|           timestamp|                 url|userId|
+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+-------+--------------------+--------------------+------+
|[1045.00502653444...| other|[162, 108, 66, 14...|  0|[49.1390526449582...|[166.576502984078...|[0.59193083225480...|running|[1630938389, 1637...|https://example.c...|   332|
|[2946.96314412445...| other|[121, 98, 168, 13...|  1|[-33.076971821503...|[134.795814986874...|[23.1945693137552...|running|[1631979990, 1649...|https://example.c...|   567|
|[349.632634314160...|  male|[177, 139, 150, 1...|  2|[85.8509495785931...|[-38.668059705926...|[38.3069744362893...|running|

In [0]:
# OverView Of DataFrame